# StructureDCA on protein–protein interactions

StructureDCA can build models for protein–protein interactions (PPIs) by adjusting the contact map or/and using a concatenated MSA (cMSA).  
In this case, the input data must be consistent between the MSA/cMSA, the 3D structure, and the chain identifiers in the correct order.

We describe here two features:

- For **heteromeric PPIs**: a StructureDCA model can be built from a cMSA if the two proteins are highly co-evolved. The user should provide the structure containing both proteins and specify the chain identifiers in the same order as in the cMSA.

- For **homomeric PPIs**: some residue contacts arise from inter-chain homomeric interactions. StructureDCA can account for these contacts. The user should provide the structure containing the homomeric complex and specify which chains correspond to the same protein.

We illustrate this with the antitoxin–toxin PPI ParE–ParD.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from structuredca import StructureDCA

# Input data
CONCATENATED_MSA_PATH = '../test_data/cMSA_ParE-ParD.a3m'
PDB_PATH = '../test_data/5ceg_8mer.pdb'
PARD_SEQUENCE_LENGTH = 93

### Manage heteromeric PPIs:

Here is how StructureDCA handles a heteromeric PPI by using a concatenated cMSA and the corresponding 3D structure.

In [ ]:
# WRONG EXAMPLE 1 --------------------------------------------------------------

# Here is an incorrect StructureDCA model
# The provided cMSA represents two proteins (ParD and ParE),
# but only one structural chain is provided: chains='A'
_sdca_dimer_wrong1 = StructureDCA(
    msa_path=CONCATENATED_MSA_PATH,
    pdb_path=PDB_PATH,
    chains='A',
    init_dca=False # early-stop the run because it makes no sense
)

# In the contact map aligned to the cMSA, the second chain is not aligned
# -> ParD (antitoxin, first chain, chain 'A') is correctly aligned
# -> ParE (toxin, second chain, chain 'D') is ignored
_sdca_dimer_wrong1.alignment.show() # show cMSA vs. PDB chains alignment -> incorrect
plt.imshow(_sdca_dimer_wrong1.contacts, cmap="Blues") # show the contact map as aligned to the cMSA
plt.title("Wrong Alignment 1: chains='A'")
plt.axvline(PARD_SEQUENCE_LENGTH, color="black")
plt.axhline(PARD_SEQUENCE_LENGTH, color="black")
plt.show()
plt.clf()

# WRONG EXAMPLE 2 --------------------------------------------------------------

# Here is a second incorrect StructureDCA model
# The two chains provided are in the wrong order
_sdca_dimer_wrong2 = StructureDCA(
    msa_path=CONCATENATED_MSA_PATH,
    pdb_path=PDB_PATH,
    chains='DA',
    init_dca=False # early-stop the run because it makes no sense
)

# In the contact map aligned to the cMSA, the chains are incorrectly aligned
# -> ParD (antitoxin, first chain, chain 'A') is ignored
# -> ParE (toxin, second chain, chain 'D') is aligned
_sdca_dimer_wrong2.alignment.show() # show cMSA vs. PDB chain alignment -> incorrect
plt.imshow(_sdca_dimer_wrong2.contacts, cmap="Blues") # show the contact map as aligned to the cMSA
plt.title("Wrong Alignment 2: chains='DA'")
plt.axvline(PARD_SEQUENCE_LENGTH, color="black")
plt.axhline(PARD_SEQUENCE_LENGTH, color="black")
plt.show()
plt.clf()

# CORRECT EXAMPLE --------------------------------------------------------------

# The two chains in the cMSA are ordered ParD–ParE, which corresponds to chains='AD'
sdca_dimer = StructureDCA(
    msa_path=CONCATENATED_MSA_PATH,
    pdb_path=PDB_PATH,
    chains='AD',
)

# In the resulting contact map aligned to the cMSA, both chains are correctly aligned
# -> we observe the contact maps of both ParD and ParE
# -> we also observe inter-chain residue contacts
# -> some residues are missing in the structure, but the aligner handles them well
sdca_dimer.alignment.show() # show cMSA vs. PDB chain alignment -> correct
plt.imshow(sdca_dimer.contacts, cmap="Blues") # show the contact map as aligned to the cMSA
plt.title("Correct Alignment: chains='AD'")
plt.axvline(PARD_SEQUENCE_LENGTH, color="black")
plt.axhline(PARD_SEQUENCE_LENGTH, color="black")
plt.show()
plt.clf()

### Manage homomeric PPIs:

In addition, the provided 3D structure (PDB `5ceg`, Biological Unit 1) represents the octameric conformation (4 ParD and 4 ParE chains) organized as four copies of ParD–ParE dimer units.  
In the previous run, we ensured that the two provided chains correspond to one ParD–ParE dimer unit (chain `A` for ParD and chain `D` for ParE).

However, we can also include contacts that arise from interfaces between different dimer units.  
In this case, the user must specify which chains in the 3D structure correspond to the same protein.  
In the provided structure, ParE chains are `A`, `C`, `E` and `G` and ParD chains are `D`, `B`, `H` and `F`.  
Homomeric protein groups should be separated by `:`, therefore the syntax for the StructureDCA parameter is `homomeric_chains='ACEG:DBHF'`.

**Note:** in simple cases, when the target chain is a single protein (`chains='A'`) and the MSA is a standard MSA, one can simply provide `homomeric_chains='ACEG'`.

In [ ]:
# StructureDCA for the octamer 4x(ParD–ParE)
# The two chains in the cMSA are ordered ParD–ParE, which corresponds to chains='AD'
# In addition, we include contacts between different dimer units
sdca_octamer = StructureDCA(
    msa_path=CONCATENATED_MSA_PATH,
    pdb_path=PDB_PATH,
    chains='AD',
    homomeric_chains='ACEG:DBHF',
)

# The resulting contact map also includes contacts between different dimer units
sdca_octamer.alignment.show() # the alignment is exactly the same as previously
contacts_dimer = sdca_dimer.contacts.astype(np.int32)
contacts_octamer = sdca_octamer.contacts.astype(np.int32)
plt.imshow(contacts_dimer + contacts_octamer, cmap="Blues") # show the contact map aligned to the cMSA
plt.title((
    "Correct alignment: chains='AD' + homomeric_chains='ACEG:DBHF'\n"
    " - dark blue: intra ParD-ParE dimer contacts (chains='AD')\n"
    " - light blue: inter-dimer contacts"
))
plt.axvline(PARD_SEQUENCE_LENGTH, color="black")
plt.axhline(PARD_SEQUENCE_LENGTH, color="black")
plt.show()
plt.clf()
